In [2]:
!pip install -q -U transformers accelerate bitsandbytes
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rsna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 61.7 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
from google.colab import files
files.upload()   # train.csv

Saving train.csv to train.csv


{'train.csv': b'StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker\'s,Contusion,Fracture\n1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260,T\xc3\xa9cnica: RMN de la rodilla. Resultados: Rotura de menisco interno. Signo de necrosis avascular subcondral en el c\xc3\xb3ndilo femoral medial. Artrosis femorotibial medial. Derrame. . Impresi\xc3\xb3n: Rotura de menisco interno. Signo de necrosis avascular subcondral en el c\xc3\xb3ndilo femoral medial. Artrosis femorotibial medial. Derrame.,,,,,,,,,,,,\n1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585,"[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindingen:",,,,,,,,,,,,\n1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231,"Hallazgos:\nNo hay alte

In [4]:
%%writefile colab_labels.py
import re, json, gc, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

STUDY_CSV = "/content/train.csv"
OUTDIR    = "/content/drive/MyDrive/rsna"   # Drive -> kalici
MODELS = [
    ("14b", "Qwen/Qwen2.5-14B-Instruct", False),
    ("32b", "Qwen/Qwen2.5-32B-Instruct", True),
]
BATCH, MAXLEN = 16, 1024

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEYS = ["ACL","MCL","MedMeniscus","LatMeniscus","MedOA","LatOA","PFOA",
        "Effusion","Synovitis","Baker","Contusion","Fracture"]
K2L = dict(zip(KEYS, LABEL_COLS))

SYS = (
    "You are a radiology report labeler for knee MRI. Reports may be in any language. "
    "For each finding output 1 if PRESENT/abnormal, else 0. Respect negations "
    "(no/sin/geen/kein/without/normal/intact = 0). If not mentioned, 0.\n"
    "ACL=anterior cruciate ligament tear; MCL=medial collateral ligament injury; "
    "MedMeniscus=medial meniscus tear; LatMeniscus=lateral meniscus tear; "
    "MedOA=medial compartment OA/cartilage loss; LatOA=lateral compartment OA; "
    "PFOA=patellofemoral OA/chondromalacia; Effusion=joint effusion; "
    "Synovitis=synovitis; Baker=Baker's/popliteal cyst; "
    "Contusion=bone contusion/bone marrow edema; Fracture=fracture.\n"
    'Answer ONLY compact JSON: {"ACL":0,"MCL":0,"MedMeniscus":0,"LatMeniscus":0,"MedOA":0,'
    '"LatOA":0,"PFOA":0,"Effusion":0,"Synovitis":0,"Baker":0,"Contusion":0,"Fracture":0}'
)

def parse(text):
    out = {c: 0 for c in LABEL_COLS}
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    if not m: return out
    try: d = json.loads(m.group(0))
    except Exception: return out
    for k, lab in K2L.items():
        out[lab] = 1 if d.get(k, 0) in (1, "1", True, "yes", "true") else 0
    return out

def run_model(model_id, use_4bit, reports):
    tok = AutoTokenizer.from_pretrained(model_id); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(device_map="auto", dtype=torch.float16)
    if use_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    order = sorted(range(len(reports)), key=lambda k: len(reports[k]))
    res = [None]*len(reports)
    for i in range(0, len(order), BATCH):
        idxs = order[i:i+BATCH]
        msgs = [[{"role":"system","content":SYS},
                 {"role":"user","content":f"Report:\n{reports[k][:2500]}"}] for k in idxs]
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=110, do_sample=False, pad_token_id=tok.pad_token_id)
        gen = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        for k, g in zip(idxs, gen): res[k] = parse(g)
        print(f"  {min(i+BATCH,len(order))}/{len(order)}", flush=True)
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(res, columns=LABEL_COLS)

def f1_report(preds_df, ids, gold):
    gp = pd.concat([ids.reset_index(drop=True), preds_df], axis=1).set_index("StudyInstanceUID")
    rows = []
    for c in LABEL_COLS:
        y = gold[c].astype(float).values; p = gp.loc[gold["StudyInstanceUID"], c].values
        tp=((p==1)&(y==1)).sum(); fp=((p==1)&(y==0)).sum(); fn=((p==0)&(y==1)).sum()
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        rows.append(2*pr*rc/(pr+rc) if pr+rc else 0)
    return float(np.mean(rows))

def main():
    df = pd.read_csv(STUDY_CSV)
    reports = df["Report"].fillna("").astype(str).tolist()
    gold = df[df[LABEL_COLS].notna().any(axis=1)].reset_index(drop=True)
    preds = {}
    for name, mid, q4 in MODELS:
        print(f"\n[i] {name} ({mid}) calisiyor...", flush=True)
        p = run_model(mid, q4, reports)
        pd.concat([df[["StudyInstanceUID"]], p], axis=1).to_csv(f"{OUTDIR}/weak_labels_{name}_full.csv", index=False)
        preds[name] = p
        f1 = f1_report(p, df[["StudyInstanceUID"]], gold)
        print(f"[OK] {name} makro F1 (58 altin): {f1:.3f}", flush=True)
    a, b = preds["14b"], preds["32b"]
    cmp = a.where(a.values == b.values, b)
    pd.concat([df[["StudyInstanceUID"]], cmp], axis=1).to_csv(f"{OUTDIR}/weak_labels_cmp_full.csv", index=False)
    f1c = f1_report(cmp, df[["StudyInstanceUID"]], gold)
    agree = (a.values == b.values).mean()
    print(f"\n=== KARSILASTIRMALI ===", flush=True)
    print(f"14b-32b anlasma orani: {agree:.3f}", flush=True)
    print(f"cmp makro F1 (58 altin): {f1c:.3f}   (7B: 0.678)", flush=True)
    print(f"Kaydedildi: {OUTDIR}/ (14b_full, 32b_full, cmp_full)", flush=True)

if __name__ == "__main__":
    main()

Writing colab_labels.py


In [5]:
!python -u colab_labels.py


[i] 14b (Qwen/Qwen2.5-14B-Instruct) calisiyor...
config.json: 100% 663/663 [00:00<00:00, 3.84MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 15.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 104MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 118MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 148MB/s]
model.safetensors.index.json: 100% 47.5k/47.5k [00:00<00:00, 110MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.00G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.99G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/11.9G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/19.9G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/25.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/29.5G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/29.5G

In [6]:
from google.colab import files
files.upload()   # kaggle.json (Downloads'tan, isim farketmez)

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"erenvoo","key":"0d893663a61563bc5d40beda54443ec1"}'}

In [7]:
import os, glob, shutil
src = next(iter(glob.glob("/content/kaggle*.json")), None); print("json:", src)
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(src, "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
!rm -rf /content/p1
!kaggle kernels output erenvoo/rsna-knee-pre256-p1 -p /content/p1
!echo "npy sayisi:"; ls /content/p1/train_pre/volumes 2>/dev/null | wc -l

json: /content/kaggle.json
Output file downloaded to /content/p1/train_pre/manifest.csv
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10001344172545964778532427094848882081.npy
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10001478771378979438868923030424928430.npy
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10002649128333294739274777073688197429.npy
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10002940759939066728493680169552264940.npy
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10002978947438170388686989793017172190.npy
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10003074389531472338522226501097852466.npy
Output file downloaded to /content/p1/train_pre/volumes/1.2.826.0.1.3680043.8.498.10003271121171039993431767719983720312.npy
Output file downloaded to /content/p1